# Step 1 コーパスとは何か — 設計・代表性・v1 の診断

> **用語の予習・復習**: `docs/glossary.md` の Step 1 を参照。キーワードを見て自分で説明してみてから読むこと。

> **「回」ではなく「Step」と呼ぶ理由**
>
> 本授業は8つの Step からなるが，これは8コマという意味ではない。環境構築・
> スクリプトの不具合・データの取り直しで必ず遅れが出るので，実質 12 コマ
> 程度を見込んでいる。Step は**カレンダー上の回ではなく到達点**である。
> 前の Step の成果物（XML，正規化テクスト，トークン列）ができていなければ
> 次へは進めない。逆に，できていれば何コマかかっても構わない。各 Step の
> 冒頭に到達目標を置いたのはそのためで，そこに書かれたことができているか
> どうかが，先へ進んでよいかどうかの判断基準になる。

## このステップの到達目標

1. 環境構築を完了し，`00_env_check.py` が `ALL OK` を返す
2. 「コーパス＝母集団からの標本」という見方を身につける
3. 既存コーパス（v1, 64点）の**偏り**を自分の手で数え，図にする
4. 3つの重大な欠陥（重複・外字欠落・奥付混入）を自分で再発見する

## 導入：なぜ「代表性」から始めるのか

近代日本文学の文体変化を量的に記述したい，とする。このとき私たちが本当に
測りたいのは **母集団**（1868–1960年に日本語で書かれた文学テクストの全体）の
性質である。しかし手元にあるのは **標本**（青空文庫にあり，著作権が切れ，
誰かが入力した64点）にすぎない。

標本が母集団を歪んだ形で代表していると，どんなに精緻なモデルを当てても
**その歪みを測ることになる**。

> 「昭和期の小説は語彙が平易になった」という結論が出たとする。
> しかしコーパスの昭和期7点が児童向け読物だったら？

本授業ではこの問いを最初に置く。パイプラインの前半 4 ステップはすべて，
**分析に耐える標本を作る**ための工程である。

## 参考

- Biber, D. (1993) Representativeness in corpus design. *LLC* 8(4).
- 田野村忠温 (2011)「コーパスとコーパス言語学」『日本語学』
- 前川喜久雄 (2013)『コーパス入門』（講座日本語コーパス1）朝倉書店


In [ ]:
# ---- 共通の準備（毎回このセルから実行する）----------------------------
import os, sys, csv, json, math, random, shutil, subprocess, warnings
import importlib.util
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore', category=FutureWarning)

# リポジトリのルートを自動で探す（notebooks/ から1つ上）
ROOT = Path.cwd()
while not (ROOT / 'config' / 'pipeline.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
print('ROOT =', ROOT)

# 日本語フォント（□ にならないように）
for cand in ['Hiragino Sans', 'Yu Gothic', 'Meiryo',
             'Noto Sans CJK JP', 'IPAexGothic', 'MS Gothic']:
    if cand in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = cand
        break
else:
    print('[!] 日本語フォントが見つかりません。docs/00_setup_students.md §1.7 を参照。')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ---- 図はすべて SVG（ベクタ）で保存する ---------------------------------
# 論文・スライドに載せる図は拡大しても劣化してはならない。PNG は解像度が
# 固定されるので，投影や印刷で文字が潰れる。SVG なら任意の倍率で鮮明で，
# Illustrator / Inkscape で軸ラベルだけを直すこともできる。
FIG_EXT   = 'svg'
RASTER_DPI = 200          # rasterized=True の要素にだけ効く
plt.rcParams['svg.fonttype']       = 'path'   # 文字をアウトライン化して環境非依存に
plt.rcParams['savefig.transparent'] = False
# 画面へのインライン表示は既定（PNG）のままにする。
# InlineBackend.figure_formats を 'svg' に変えると，JupyterLab や
# VS Code の版によっては図がまったく表示されなくなることがある。
# **保存されるファイルは SVG** なので，論文・スライドに使うほうは
# ベクタで手元に残る。画面で拡大して見たいときは save_fig が表示する
# パスの .svg をブラウザで開くこと。

def need(path, hint=''):
    """必要な入力があるか確かめる。無ければ**理由を表示して** False を返す。

    セルを `if p.exists():` で囲むと，入力が無いときに何も起きない。
    学生には「壊れている」と「まだ前の工程を走らせていない」の区別が
    つかず，図が出ないという相談の大半がこれである。必ず理由を出す。
    """
    p = Path(path)
    try:
        ok = p.is_file() or (p.is_dir() and any(p.iterdir()))
    except OSError:
        ok = False
    if not ok:
        print(f'[未実行] {p} がありません。')
        if hint:
            print(f'         {hint}')
        print('         この Step の前のセルを上から順に実行すること。'
              '\n         それでも出ない場合は，前の Step のノートブックが'
              '最後まで通っているか確認する。')
    return ok


def load_meta(path=None, analysis_only=True):
    """メタデータを読む。既定では**分析に使う行だけ**を返す。

    落とすのは2種類。書誌としては残すが，集計に足してはいけない行である。
      superseded … v1 の合本。増補で分冊ごとに取り直したので，足すと
                   同じ作品を二重に数える
      merged     … 分冊。03b で canonical の巻に本文を統合したので，
                   この行はもう本文を持たない（『夜明け前』『家』）
      too_short  … 1チャンクにも満たず，チャンク単位の分析に乗らない

    生の表がほしいときは ``analysis_only=False``。
    """
    df = pd.read_csv(path or META)
    if analysis_only and 'completeness' in df.columns:
        drop = df['completeness'].isin(['superseded', 'merged', 'too_short'])
        if drop.any():
            names = '，'.join(df.loc[drop, 'title_aozora'].astype(str))
            print(f'[meta] 分析から除外 {int(drop.sum())} 行: {names}')
        df = df[~drop].reset_index(drop=True)
    return df


def w_ljust(text, width):
    """全角を2桁と数えて左詰めする。

    ``f'{s:<26}'`` は**文字数**で詰めるので，日本語の作品名を並べると
    桁が揃わない（全角は2桁ぶんの幅を占める）。表として読ませるなら
    表示幅で詰めること。

    **なお，一覧を出すなら ``show()`` で表にするほうがよい**（下記）。
    この関数は，表にしにくいもの（KWIC の前後文脈など）を print で
    並べるときに使う。
    """
    import unicodedata
    text = str(text)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in text)
    return text + ' ' * max(0, width - w)


# ---------------------------------------------------------------------------
# 分析結果の表示
# ---------------------------------------------------------------------------
# **一覧は print ではなく表で出す。**
#   * print は桁が揃わない（全角の幅）。数字の比較がしにくい
#   * 列に名前が付かないので，あとで見返したときに何の数字か分からない
#   * 並べ替えも絞り込みもできない
# 表にすると，列名がそのまま「何を測ったか」の記録になる。
# **ただし何でも表にするのではない。** 単発の数値・警告・KWIC の前後文脈は
# 文のほうが読みやすい。目安は「2列以上あるか」「行が並ぶか」。
TABLE_STYLES = [
    {'selector': 'caption',
     'props': [('caption-side', 'top'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '0 0 .4em 0'),
               ('color', '#33322e'), ('font-size', '95%')]},
    {'selector': 'th',
     'props': [('background', '#f2f2ef'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '.26em .7em'),
               ('border-bottom', '1px solid #c6c5bd'), ('white-space', 'nowrap')]},
    {'selector': 'td',
     'props': [('padding', '.22em .7em'), ('border-bottom', '1px solid #ecebe6')]},
    {'selector': 'tbody tr:hover td', 'props': [('background', '#f7f7f4')]},
]


def show(df, caption='', fmt=None, index=False, header=True, na='—', align=None):
    """DataFrame を表として表示する。Jupyter 以外でも落ちない。

    ``fmt`` は pandas の ``Styler.format`` に渡す辞書
    （例 ``{'一致率': '{:.1%}', 'G²': '{:.0f}'}``）。
    数値の列は自動で右寄せにする。``align`` で列ごとに寄せを指定できる。
    KWIC の左文脈を ``align={'左文脈': 'right'}`` にすると，
    **キーワードが縦に揃う**（等幅フォントに頼らずに揃う）。
    返り値は ``df`` なので ``t = show(df)`` として続けて使える。
    """
    if isinstance(df, pd.Series):
        df = df.to_frame()
    try:
        from IPython.display import display as _display
        st = df.style.format(fmt, na_rep=na) if fmt else df.style.format(na_rep=na)
        st = st.set_table_styles(TABLE_STYLES)
        num = list(df.select_dtypes('number').columns)
        if num:
            st = st.set_properties(subset=num, **{'text-align': 'right'})
        for col, side in (align or {}).items():
            if col in df.columns:
                st = st.set_properties(subset=[col],
                                       **{'text-align': side,
                                          'white-space': 'pre'})
        if caption:
            st = st.set_caption(caption)
        if not index:
            st = st.hide(axis='index')
        if not header:
            st = st.hide(axis='columns')
        _display(st)
    except Exception:                                   # noqa: BLE001
        # ノートブックの外（スクリプトから import したとき）でも読める形
        if caption:
            print(caption)
        print(df.to_string(index=index, header=header))
    return df


def grid(items, ncol=8, caption=''):
    """語の並びを ``ncol`` 列の表にして表示する。

    40 語を1行に流すと折り返しで読めない。列に切ると目で追える。
    順位が要るなら ``show()`` に順位列を付けた表を渡すこと。
    """
    items = [str(x) for x in items]
    rows = [items[i:i + ncol] for i in range(0, len(items), ncol)]
    rows = [r + [''] * (ncol - len(r)) for r in rows]
    t = pd.DataFrame(rows, columns=[f'_{i}' for i in range(ncol)])
    return show(t, caption=caption, header=False)


def work_rows(meta_df=None):
    """``work_stem`` からメタデータの行を引く辞書を作る。

    ``meta_df`` を省くと**分析対象外の行も含めた全件**から作る。
    表示用の名前は，分析から外した作品についても引けるほうがよい。

    **鍵の綴りに注意。** 青空文庫の作品 ID は索引では 0 埋めされていない
    （``1743``）が，本パイプラインのファイル名は6桁に 0 埋めしてある
    （``000119_001743``）。素朴に連結すると ``000119_1743`` となり，
    **1件も一致しない**。辞書は空振りしても例外を出さないので，
    誰の何だか分からないまま最後まで通ってしまう。両方の綴りを登録する。

    ``file_v1`` は増補 45 点では空である。``os.path.splitext(nan)`` は
    例外になるので，文字列であることを確かめてから使う。
    """
    if meta_df is None:
        meta_df = load_meta(analysis_only=False)
    d = {}
    for _, r in meta_df.iterrows():
        fv = r.get('file_v1')
        if isinstance(fv, str) and fv.strip():
            d[os.path.splitext(fv)[0]] = r
        pid = str(r.get('aozora_person_id') or '').strip()
        wid = str(r.get('aozora_work_id') or '').strip()
        if pid and wid and pid.lower() != 'nan' and wid.lower() != 'nan':
            for k in (f'{pid.zfill(6)}_{wid.zfill(6)}',
                      f'{pid}_{wid}', f'{pid.zfill(6)}_{wid}'):
                d[k] = r
    return d


def work_labels(meta_df=None, maxlen=12, with_year=False):
    """``work_stem`` → ``作者『作品』`` の対応表を返す。

    ``000119_001743`` と出されても誰の何だか分からない。距離の近い
    ペアを見るときに**どの作家のどの作品か**が分からなければ，
    「作家効果か時代効果か」という問いにそもそも答えられない。
    表示するときは必ずこれを通すこと。
    """
    out = {}
    for k, r in work_rows(meta_df).items():
        t = str(r.get('title_aozora') or '')
        lab = f"{r.get('author_ja', '?')}『{t[:maxlen]}』"
        if with_year and str(r.get('year_first') or '').strip():
            lab += f"({r['year_first']})"
        out[k] = lab
    return out


def attach_meta(df, cols, stem_col='work_stem', meta_df=None, quiet=False,
                fill_blank=True):
    """``df`` に足りないメタデータの列を，``work_stem`` から引いて補う。

    ``fill_blank=True``（既定）なら，**列はあるのに値が空**のセルも補う。
    列が無いより，列があって半分が空のほうが危ない。列が無ければ
    ``AttributeError`` で止まるが，値が空だと**図がそのまま描けてしまう**。
    2026-09-22 に 07 の突合が外れ，101 点のうち 62 点の ``year_first`` が
    空になった。図は描けたが，62 点が「初出年不明」の灰色で並んだ。
    値の空きも数えて報告し，ここで補えるものは補う。

    分析スクリプトの出力は，その分析に要る列しか書かない。
    ``09_doc2vec.py`` の ``work_vectors.csv`` に ``genre_main`` が無いのは
    その一例である。ノートブックで ``wv.genre_main`` と書けば
    ``AttributeError: 'DataFrame' object has no attribute 'genre_main'``
    になるが，**足りないのは列であって情報ではない**。
    メタデータ表には必ずあるのだから，ここで引いて補えばよい。

    出力 CSV の列構成に図の描画が依存するのは弱い。分析スクリプトを
    書き換えるたびに図が落ちる。図の側で「要る列を宣言して取りに行く」
    ほうが，どちらを先に走らせても通る。

    引けなかった列は空のまま残し ``[warn]`` を出す。図が落ちるより，
    「この軸は塗れなかった」と分かったうえで出るほうがよい。
    """
    df = df.copy()
    if stem_col not in df.columns:
        if not quiet:
            print(f'[warn] {stem_col} 列が無いので補完できない: {list(cols)}')
        for c in cols:
            if c not in df.columns:
                df[c] = ''
        return df

    rows = work_rows(meta_df)

    # **まず鍵が合っているかを見る。** 合っていなければ何も補えない。
    # 「1件も合わない」のはたいてい 0 埋めの綴り違いで，黙って通すと
    # 全部の軸が空のまま図になる。
    stems = df[stem_col].astype(str)
    found = stems.map(lambda s: s in rows)
    if not quiet and not found.all():
        n_miss = int((~found).sum())
        lv = 'FATAL' if found.sum() == 0 else 'warn '
        print(f'[{lv}] {stem_col} がメタデータと突合できない行が '
              f'{n_miss}/{len(df)} 件ある: '
              + '，'.join(stems[~found].head(4)))
        if found.sum() == 0:
            print('        **1件も合っていない。** 作品 ID の 0 埋めの'
                  '綴り違いを疑うこと（例 000119_1743 と 000119_001743）。')
            print('        この表を作ったスクリプトの鍵の作り方を直すこと。')

    def _blank(v):
        return v is None or str(v).strip().lower() in ('', 'nan', 'none')

    for c in cols:
        if c not in df.columns:
            vals = [(lambda r: '' if r is None or _blank(r.get(c))
                     else r.get(c))(rows.get(s)) for s in stems]
            df[c] = vals
            n = int(sum(1 for v in vals if str(v).strip()))
            if not quiet:
                mark = 'ok  ' if n == len(df) else 'warn'
                print(f'[{mark}] {c} をメタデータから補完: {n}/{len(df)} 件')
            continue

        if not fill_blank:
            continue
        # 列はある。空のセルだけを埋める。
        blank = df[c].map(_blank)
        if not blank.any():
            continue
        filled = 0
        vals = df[c].tolist()
        for i, (s, is_blank) in enumerate(zip(stems, blank)):
            if not is_blank:
                continue
            r = rows.get(s)
            if r is not None and not _blank(r.get(c)):
                vals[i] = r.get(c)
                filled += 1
        df[c] = vals
        if not quiet:
            left = int(sum(1 for v in vals if _blank(v)))
            mark = 'fix ' if left == 0 else 'warn'
            print(f'[{mark}] {c} は {int(blank.sum())}/{len(df)} 件が空だった'
                  f' → {filled} 件をメタデータから補完'
                  + ('' if left == 0 else f'（なお {left} 件が空）'))
            if left:
                print('        **その列で塗る図・集計は，この件数を'
                      '報告に書くこと。**')
    return df


def label_points(ax, xs, ys, texts, fontsize=8, color='#333333', pad=4,
                 leader='line', leader_min=13, crowd_r=26,
                 leader_color='#8a8a83'):
    """散布図の注記を，重ならない位置だけに置き，遠いものは引き出し線で結ぶ。

    素朴に ``ax.annotate(t, (x, y))`` と書くと，**注目すべき点ほど一箇所に
    固まる**ので注記が必ず重なって読めなくなる。文語標識の上位は
    どれも口語標識がほぼ 0 で，対数軸の右下隅に密集するのが典型である。

    そこで点の周囲を順に試し，他の注記とも他の点とも重ならず，かつ軸の
    内側に収まる位置があればそこに置く。どこにも置けない注記は**置かずに
    数だけ報告する**。読めない字を重ねるより，図の外（下の表）で番号から
    引くほうがよい。

    **離れた位置に置いた注記は，引き出し線で点と結ぶ。** 避けた結果として
    注記は点から離れるので，線が無いとどの点の名前なのか分からなくなる
    ——密集した領域では隣の点の名前だと読まれる。線があれば，遠くへ逃がす
    ことに副作用が無くなるので，**近くに空きが無い注記も置ける**ようになる
    （候補の輪を 24・30 ポイントまで広げてあるのはそのため）。

    ``leader``
        ``'line'``（既定）… 矢じりの無い細線で結ぶ。図版の慣例はこちら。
        6.5pt の文字に矢じりを付けると，印そのものを覆って点が読めなくなる
        ``'arrow'`` … 小さな矢じりを付ける
        ``'none'`` … 結ばない（従来どおり）
    ``leader_min``
        この距離（ポイント）より遠くに置いた注記を結ぶ。既定は 0，
        つまり**すべて結ぶ**。注記は必ず点から離れた位置に置かれるので，
        離れている以上「どの点の名前か」は線でしか確定しない。
        線を省くと，隣の点の名前だと読まれる余地が残る。
    ``crowd_r``
        注記の近くに**自分以外の点**がこの半径（ピクセル）内にあるかを
        見る。``leader_min`` を上げて線を減らしたときでも，
        紛れる相手が居る注記だけは必ず結ぶための保険である。

    表示座標で矩形の重なりを見るので，**軸の位置が確定してから**呼ぶ。
    ``fig.tight_layout()`` はこの関数より**前**に呼ぶこと（後で呼ぶと軸が
    動き，せっかく避けた位置がずれる）。戻り値は置けた注記の数。
    """
    from matplotlib.transforms import Bbox
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    texts = list(texts)
    if len(texts) == 0:
        return 0
    # **長さが違えば黙って切り詰めずに止める。** zip は短いほうに合わせるので，
    # 座標だけを絞り込んで名前を絞り忘れると，先頭から順に**別の作品の名前**が
    # 貼られた図が，何の警告も出さずに出来上がる。これがいちばん重い事故である。
    if not (len(xs) == len(ys) == len(texts)):
        raise ValueError(
            f'label_points: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'名前={len(texts)}）。座標と名前を同じ添字で絞り込むこと。'
            'たとえば P[pick,0] と組むのは names[pick] であって names ではない。')
    fig = ax.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    axbb = ax.get_window_extent(renderer=ren)

    def pad_box(b, w=2.0, h=1.5):
        """注記の矩形に**絶対量の余白**を足す。

        倍率（``expanded(1.08, …)``）では足りない。1桁の数字は幅 8px ほど
        なので 8% は 0.6px にしかならず，隣り合う注記が触れるほど近くても
        「重なっていない」と判定される。**``21`` が「21」と読める**のは
        これが原因である。文字の大小によらず一定の余白を確保する。
        """
        return Bbox.from_extents(b.x0 - w, b.y0 - h, b.x1 + w, b.y1 + h)

    def bb_of(ann):
        # Annotation 自身の get_window_extent を使うこと。
        # Text.get_window_extent(ann, ...) を呼ぶと xy の位置が無視され，
        # xytext のオフセットを絶対座標と見た矩形が返って判定が壊れる。
        return ann.get_window_extent(renderer=ren)

    blocked = []
    for coll in ax.collections:
        try:
            for p in coll.get_offsets():
                px, py = ax.transData.transform(p)
                blocked.append(Bbox.from_bounds(px - pad, py - pad,
                                                2 * pad, 2 * pad))
        except Exception:                                    # noqa: BLE001
            pass

    # **まっすぐ真上・真下を先に試す。** 点の直上に中央揃えで置ければ，
    # それがいちばん素直で，引き出し線も要らない。横へずらすのは，
    # 直上が塞がっていたときの次善である。
    #
    # 横へずらす輪は 12 ポイントから始める。**線が線として見える長さを
    # 確保する**ため。8 ポイントに置くと引き出し線が3ピクセルの点にしか
    # ならず，汚れと区別がつかない。
    # **真上に置けなければ，まず真上へ逃がす。** 横へ逃がすと注記の左右の
    # 順序が点の順序と入れ替わり，引き出し線も交差する。真上に段を重ねる
    # 限り，x は動かないので順序は必ず保たれる。横へずらすのは最後。
    # **横のずらし幅は小さく取る。** 横へ 30 ポイントも動かすと，注記が
    # 隣の点の真上に乗り，引き出し線で結んでも読みにくい。真上に段を
    # 重ねるほうが先で（x が動かないので順序が保たれる），横は 8→18
    # ポイントの範囲に収める。
    CAND = [(0, 9), (0, -11), (0, 20), (0, -22), (0, 31), (0, -33),
            (8, 5), (-8, 5), (8, -12), (-8, -12),
            (11, 0), (-11, 0),
            (13, 9), (-13, 9), (13, -16), (-13, -16),
            (18, 0), (-18, 0), (18, 14), (-18, 14),
            (0, 42), (0, -44)]

    def _ha(dx):
        # dx が 0 なら**中央揃え**。ここを 'left' にすると，真上に置いた
        # つもりの注記が文字幅の半分だけ右にずれ，隣の点の上に乗る。
        return 'center' if dx == 0 else ('left' if dx > 0 else 'right')
    placed, chosen, skipped = [], [], 0
    for x, y, t in zip(xs, ys, texts):
        # **自分が指している点は避けない。** 除かないと，注記は必ず
        # 自分の点の隣に来るので全部「重なる」と判定され，1つも置けない。
        ox, oy = ax.transData.transform((x, y))
        near = [b for b in blocked
                if not (abs((b.x0 + b.x1) / 2 - ox) < 1
                        and abs((b.y0 + b.y1) / 2 - oy) < 1)]
        for dx, dy in CAND:
            ann = ax.annotate(str(t), (x, y), textcoords='offset points',
                              xytext=(dx, dy), fontsize=fontsize, color=color,
                              ha=_ha(dx),
                              va='bottom' if dy >= 0 else 'top', zorder=6)
            bb = pad_box(bb_of(ann))
            inside = (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                      and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1)
            if inside and not any(bb.overlaps(b) for b in placed + near):
                placed.append(bb)
                chosen.append((ann, x, y, str(t), dx, dy))
                break
            ann.remove()
        else:
            skipped += 1

    # ---- 交差をほどく ----------------------------------------------------
    # **引き出し線が交差すると，注記の左右の順序が点の順序と入れ替わる。**
    # 文語標識の上位のように順位そのものが意味を持つ図では，2 と 3 が
    # 入れ替わって並ぶだけで読み違えられる。交差している2件を見つけ，
    # **位置を入れ替えて交差が解ければ入れ替える**（2-opt）。
    def _cross(p, q, r, s):
        def o(a, b, c):
            return ((b[0] - a[0]) * (c[1] - a[1])
                    - (b[1] - a[1]) * (c[0] - a[0]))
        return (((o(r, s, p) > 0) != (o(r, s, q) > 0))
                and ((o(p, q, r) > 0) != (o(p, q, s) > 0)))

    kpt = fig.dpi / 72.0

    def _seg(i):
        ann, x, y, t, dx, dy = chosen[i]
        ox, oy = ax.transData.transform((x, y))
        return (ox, oy), (ox + dx * kpt, oy + dy * kpt)

    def _set_off(i, dx, dy):
        ann, x, y, t, _, _ = chosen[i]
        ann.set_position((dx, dy))
        ann.set_ha(_ha(dx))
        ann.set_va('bottom' if dy >= 0 else 'top')
        chosen[i] = (ann, x, y, t, dx, dy)

    def _fits(i, bb):
        ox, oy = ax.transData.transform((chosen[i][1], chosen[i][2]))
        if not (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1):
            return False
        return not any(bb.overlaps(b) for b in blocked
                       if abs((b.x0 + b.x1) / 2 - ox) > 1
                       or abs((b.y0 + b.y1) / 2 - oy) > 1)

    swaps = 0
    for _ in range(3):
        improved = False
        for i in range(len(chosen)):
            for j in range(i + 1, len(chosen)):
                if not _cross(*_seg(i), *_seg(j)):
                    continue
                di, dj = chosen[i][4:6], chosen[j][4:6]
                _set_off(i, *dj)
                _set_off(j, *di)
                bi = pad_box(bb_of(chosen[i][0]))
                bj = pad_box(bb_of(chosen[j][0]))
                others = [b for k2, b in enumerate(placed) if k2 not in (i, j)]
                good = (not bi.overlaps(bj)
                        and not any(bi.overlaps(b) or bj.overlaps(b)
                                    for b in others)
                        and _fits(i, bi) and _fits(j, bj)
                        and not _cross(*_seg(i), *_seg(j)))
                if good:
                    placed[i], placed[j] = bi, bj
                    swaps += 1
                    improved = True
                    continue
                _set_off(i, *di)
                _set_off(j, *dj)

                # 入れ替えが収まらないときは，**片方を別の候補位置へ動かす**。
                # 入れ替えは2つの箱の大きさが違うと失敗しやすい（数字1桁と
                # 作者名では幅が違う）。動かすほうは箱の大きさが変わらない。
                moved = False
                for who, other in ((i, j), (j, i)):
                    d0 = chosen[who][4:6]
                    for cx, cy in CAND:
                        if (cx, cy) == tuple(d0):
                            continue
                        _set_off(who, cx, cy)
                        bw = pad_box(bb_of(chosen[who][0]))
                        rest = [b for k2, b in enumerate(placed) if k2 != who]
                        if (_fits(who, bw)
                                and not any(bw.overlaps(b) for b in rest)
                                and not _cross(*_seg(who), *_seg(other))
                                and not any(_cross(*_seg(who), *_seg(k2))
                                            for k2 in range(len(chosen))
                                            if k2 != who)):
                            placed[who] = bw
                            swaps += 1
                            moved = improved = True
                            break
                        _set_off(who, *d0)
                    if moved:
                        break
        if not improved:
            break

    # ---- 引き出し線 ------------------------------------------------------
    # **線は配置が全部決まってから付ける。** arrowprops を付けた
    # Annotation の get_window_extent は「文字＋線」の外接矩形を返すので，
    # 配置の判定に使うと自分の点と必ず重なり，1件も置けなくなる。
    n_leader = 0
    if leader in ('line', 'arrow'):
        style = '-' if leader == 'line' else '-|>'
        for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
            ox, oy = ax.transData.transform((x, y))

            def dist_to_box(px, py, b=bb):
                # 文字の矩形から点までの距離。矩形の中なら 0。
                ddx = max(b.x0 - px, 0, px - b.x1)
                ddy = max(b.y0 - py, 0, py - b.y1)
                return (ddx * ddx + ddy * ddy) ** .5

            # **素直に置けたものには線を引かない。** 点の直上（または直下）に
            # 中央揃えで載っていて，しかもその注記にいちばん近い点が自分の
            # 点であれば，どの点の名前かは見れば分かる。線はかえって邪魔
            # である。横へ逃がしたものだけを結ぶ。
            if dx == 0 and abs(dy) <= 12:
                continue            # 点の真上・真下の一段目 → 線は要らない
            # それ以外は結ぶ。**段を上げたものも結ぶ。** 一段上げた注記の
            # 真下には別の点の注記が入るので，どちらの点のものか分からなく
            # なる。横へずらしたものは言うまでもない。
            d_other = min(
                (dist_to_box((b.x0 + b.x1) / 2, (b.y0 + b.y1) / 2)
                 for b in blocked
                 if abs((b.x0 + b.x1) / 2 - ox) > 1
                 or abs((b.y0 + b.y1) / 2 - oy) > 1),
                default=float('inf'))
            if (dx * dx + dy * dy) ** .5 < leader_min and d_other >= crowd_r:
                continue
            ann.remove()
            ax.annotate(t, (x, y), textcoords='offset points',
                        xytext=(dx, dy), fontsize=fontsize, color=color,
                        ha=_ha(dx),
                        va='bottom' if dy >= 0 else 'top', zorder=6,
                        arrowprops=dict(arrowstyle=style, linewidth=.55,
                                        color=leader_color, alpha=.9,
                                        shrinkA=1.5, shrinkB=2.5,
                                        mutation_scale=7))
            n_leader += 1

    # ---- 誤読の自己点検 --------------------------------------------------
    # **注記の最寄りの点が自分の点でないものを数える。** これが
    # 「ラベルとデータ点がずれて見える」の正体である。引き出し線を
    # 引いてあれば誤読にはならないが，線を切った設定では危険なので，
    # そのときだけ警告を出す。
    risky = []
    for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
        ox, oy = ax.transData.transform((x, y))
        cx, cy = (bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2
        d_own = ((cx - ox) ** 2 + (cy - oy) ** 2) ** .5
        d_other = min(
            (((b.x0 + b.x1) / 2 - cx) ** 2 + ((b.y0 + b.y1) / 2 - cy) ** 2) ** .5
            for b in blocked
            if abs((b.x0 + b.x1) / 2 - ox) > 1 or abs((b.y0 + b.y1) / 2 - oy) > 1
        ) if len(blocked) > 1 else float('inf')
        if d_other < d_own:
            risky.append(t)
    left = sum(1 for i in range(len(chosen)) for j in range(i + 1, len(chosen))
               if _cross(*_seg(i), *_seg(j)))
    if skipped:
        print(f'[fig] 重なるため {skipped} 件の注記を省いた（表で引くこと）')
    if left:
        print(f'[warn] 引き出し線の交差が {left} 件ほどけなかった。'
              '注記の左右の順序が点の順序と食い違う。'
              '注記を短くするか，件数を減らすこと。')
    if risky:
        head = '，'.join(str(r) for r in risky[:6])
        more = f' ほか{len(risky) - 6}件' if len(risky) > 6 else ''
        if leader in ('line', 'arrow'):
            print(f'[fig] {len(risky)} 件の注記は別の点のほうが近い'
                  f'（{head}{more}）。引き出し線で結んであるので読み違えない。')
        else:
            print(f'[warn] {len(risky)} 件の注記は**別の点のほうが近い**'
                  f'（{head}{more}）。leader="none" では読み違えが起きる。')
    return len(placed)
def reserve_right(fig, frac=0.80):
    """面の外に凡例を置いた図で，**右に余白を確保する**。

    ``tight_layout()`` は面の外に置いた凡例を数えないので，そのままだと
    凡例が図の枠から出る。静止版は ``bbox_inches='tight'`` で救われるが，
    **HTML に埋め込む版は切り取らない**（切り取ると点の位置の割合が
    ずれる）ので，凡例が切れて読めなくなる。実際に切れた。

    ``tight_layout()`` の**後**，注記（``label_points``）の**前**に呼ぶ。
    """
    fig.subplots_adjust(right=frac)


def save_fig(fig, stem, out=None):
    """図を SVG で保存してパスを表示する。

    stem は拡張子なしの名前（例 'Step1_period_balance'）。
    点が数千個ある散布図は，散布図だけ rasterized=True にしておくと
    軸と文字はベクタのままファイルが軽くなる。
    """
    d = Path(out) if out else OUT
    d.mkdir(parents=True, exist_ok=True)
    path = d / f'{stem}.{FIG_EXT}'
    fig.savefig(path, format=FIG_EXT, dpi=RASTER_DPI, bbox_inches='tight')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB)')
    return path


# ---------------------------------------------------------------------------
# 対話的な図（SVG はそのまま残す）
# ---------------------------------------------------------------------------
# 散布図の点が何百個あると，注記を付けられるのはごく一部である。残りの点は
# 「どの語か」が分からないまま眺めることになる。かといって全点に名前を
# 付ければ図は読めない。
#
# そこで**同じ図から2つ出す**。
#   * ``<stem>.svg``  … 論文・配布用。これまでどおり。加筆も拡大も自由
#   * ``<stem>.html`` … 授業・探索用。SVG をそのまま埋め込み，
#                       その上に当たり判定を重ねて，指した点の語を出す
#
# **HTML は SVG を作り直さない。同じ SVG を中に入れる。** 別に描き直すと
# 図が2種類できて，どちらが正かが分からなくなる。注記（bursty な語の
# ラベル）も SVG の中にあるのでそのまま残る。
#
# 外部の JS ライブラリは使わない。CDN が塞がれた機体でも開けるようにする。
INTERACTIVE_CSS = """
:root { --ink:#1f1e1b; --ink2:#5a5a55; --line:#d8d7d0; --surface:#ffffff;
        --wash:#f7f7f4; --accent:#184f95; }
* { box-sizing:border-box; }
body { margin:0; padding:24px 16px 48px; background:var(--wash);
       color:var(--ink); font-family:"Hiragino Sans","Noto Sans JP",
       "Yu Gothic",system-ui,sans-serif; line-height:1.6; }
.wrap { max-width:1100px; margin:0 auto; }
h1 { font-size:1.15rem; margin:0 0 .2em; font-weight:650; }
.sub { color:var(--ink2); font-size:.86rem; margin:0 0 1.1em; }
.card { background:var(--surface); border:1px solid var(--line);
        border-radius:10px; padding:14px; }
.figbox { position:relative; }
.figbox svg { width:100%; height:auto; display:block; }
#hit { position:absolute; inset:0; cursor:crosshair; }
#ring { position:absolute; width:22px; height:22px; margin:-11px 0 0 -11px;
        border:2px solid var(--accent); border-radius:50%;
        pointer-events:none; opacity:0; transition:opacity .08s; }
#tip { position:absolute; z-index:5; min-width:190px; max-width:290px;
       background:var(--surface); border:1px solid var(--line);
       border-radius:8px; box-shadow:0 6px 20px rgba(0,0,0,.13);
       padding:9px 11px; font-size:.8rem; pointer-events:none; opacity:0;
       transition:opacity .08s; }
#tip .term { font-size:1.05rem; font-weight:650; letter-spacing:.02em;
             margin-bottom:.35em; word-break:break-all; }
#tip dl { display:grid; grid-template-columns:auto 1fr; gap:1px 10px;
          margin:0; }
#tip dt { color:var(--ink2); font-size:.74rem; white-space:nowrap; }
#tip dd { margin:0; text-align:right; font-variant-numeric:tabular-nums;
          font-weight:600; }
.bar { display:flex; gap:10px; align-items:center; flex-wrap:wrap;
       margin:14px 0 0; font-size:.82rem; color:var(--ink2); }
.bar input { font:inherit; padding:5px 9px; border:1px solid var(--line);
             border-radius:6px; min-width:190px; background:var(--surface); }
.bar a { color:var(--accent); }
table { border-collapse:collapse; width:100%; font-size:.78rem;
        margin-top:10px; }
th,td { padding:4px 8px; border-bottom:1px solid #ecebe6; text-align:left;
        white-space:nowrap; }
th { background:var(--wash); position:sticky; top:0; font-weight:650; }
td.num { text-align:right; font-variant-numeric:tabular-nums; }
tbody tr:hover td { background:var(--wash); }
tbody tr.on td { background:#eaf1fb; }
.scroll { max-height:340px; overflow:auto; border:1px solid var(--line);
          border-radius:8px; margin-top:10px; }
.hint { font-size:.78rem; color:var(--ink2); margin:.6em 0 0; }
#links { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#links line { stroke:var(--accent); stroke-width:1.1; opacity:.55; }
#links circle { fill:none; stroke:var(--accent); stroke-width:1.4; opacity:.8; }
#marks { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#marks circle { fill:none; stroke:#d55e00; stroke-width:1.6; opacity:.9; }
#tip .notes { margin:.45em 0 0; font-size:.76rem; color:var(--ink);
              border-top:1px solid var(--line); padding-top:.4em;
              line-height:1.5; word-break:break-all; }
#tip .notes b { color:var(--ink2); font-weight:600; }
.prov { font-size:.72rem; color:var(--ink2); margin:.9em 0 0;
        border-top:1px solid var(--line); padding-top:.6em;
        font-variant-numeric:tabular-nums; }
.danger { background:#fdf0ea; border:1px solid #e8a37c; border-radius:8px;
          padding:9px 12px; font-size:.85rem; color:#8a3b10;
          margin:0 0 12px; }
"""

INTERACTIVE_JS = r"""
// 点は data-* ではなく JSON で渡す。語はコーパス由来の任意の文字列なので，
// **HTML に文字列連結で差し込まない**（textContent で入れる）。
// 見出し（keys・nhead）は全点で同じなら1回だけ入っている。点が1万個ある
// 図では，これで HTML が 1 MB 以上軽くなる。古い形（配列だけ）も読む。
const RAW = JSON.parse(document.getElementById('pts-data').textContent);
const PTS = Array.isArray(RAW) ? RAW : RAW.pts;
const KEYS = (RAW && RAW.keys) || [];
const NHEAD = (RAW && RAW.nhead) || '';
const LINKNOTES = !!(RAW && RAW.linknotes);
function pairsOf(p) {
  if (p.fields) return p.fields;
  if (p.v) return p.v.map((x, i) => [KEYS[i] || '', x]);
  return [];
}
function notesOf(p) {
  if (p.notes) return p.notes;
  if (p.n) return [NHEAD, p.n];
  // 本文が無く linknotes が立っているときは，線で結ぶ先の語を並べる
  if (LINKNOTES && p.links && p.links.length) {
    return [NHEAD, p.links.map(j => (PTS[j] || {}).term || '').join(' ')];
  }
  return null;
}
const box = document.getElementById('hit');
const tip = document.getElementById('tip');
const ring = document.getElementById('ring');
const rows = Array.from(document.querySelectorAll('tbody tr'));
const links = document.getElementById('links');
const marks = document.getElementById('marks');

// **最も近い点を拾う。** 点の直径は数ピクセルしかないので，
// 「真上に置く」ことを要求すると誰も当てられない（dataviz の規則）。
// カーソルに最も近い点を選び，遠すぎるときだけ何も出さない。
function nearest(px, py, w, h) {
  let best = null, bd = 1e9;
  for (const p of PTS) {
    const dx = p.x * w - px, dy = p.y * h - py;
    const d = dx * dx + dy * dy;
    if (d < bd) { bd = d; best = p; }
  }
  return Math.sqrt(bd) <= 34 ? best : null;   // 34px より遠ければ出さない
}

function fill(p) {
  tip.textContent = '';
  const h = document.createElement('div');
  h.className = 'term';
  h.textContent = p.term;                     // ← 連結しない
  tip.appendChild(h);
  const dl = document.createElement('dl');
  for (const [k, v] of pairsOf(p)) {
    const dt = document.createElement('dt'); dt.textContent = k;
    const dd = document.createElement('dd'); dd.textContent = v;
    dl.appendChild(dt); dl.appendChild(dd);
  }
  tip.appendChild(dl);
  const nt = notesOf(p);
  if (nt) {                                   // 近傍語など，横に長い情報
    const n = document.createElement('p');
    n.className = 'notes';
    const b = document.createElement('b');
    b.textContent = nt[0] + ' ';
    n.appendChild(b);
    n.appendChild(document.createTextNode(nt[1]));
    tip.appendChild(n);
  }
}

// **原空間での近傍を線で結ぶ。** 画面の近さは射影の結果にすぎない。
// 線が遠くへ伸びるなら，その点の近傍関係は2次元に収まっていない。
// これを見せるのが，この図でいちばん大事なところである。
function drawLinks(p, w, h) {
  if (!links) return;
  while (links.firstChild) links.removeChild(links.firstChild);
  if (!p.links || !p.links.length) return;
  const NS = 'http://www.w3.org/2000/svg';
  for (const j of p.links) {
    const q = PTS[j];
    if (!q) continue;
    const ln = document.createElementNS(NS, 'line');
    ln.setAttribute('x1', p.x * w); ln.setAttribute('y1', p.y * h);
    ln.setAttribute('x2', q.x * w); ln.setAttribute('y2', q.y * h);
    links.appendChild(ln);
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', q.x * w); c.setAttribute('cy', q.y * h);
    c.setAttribute('r', 5);
    links.appendChild(c);
  }
}

let cur = null;
function show(p, px, py) {
  const w = box.clientWidth, h = box.clientHeight;
  if (p !== cur) { fill(p); drawLinks(p, w, h); cur = p; }
  ring.style.left = (p.x * w) + 'px';
  ring.style.top = (p.y * h) + 'px';
  ring.style.opacity = 1;
  tip.style.opacity = 1;
  // はみ出さないように寄せる
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lx = px + 16, ly = py + 14;
  if (lx + tw > w) lx = px - tw - 16;
  if (ly + th > h) ly = py - th - 14;
  tip.style.left = Math.max(0, lx) + 'px';
  tip.style.top = Math.max(0, ly) + 'px';
  rows.forEach(r => r.classList.toggle('on', r.dataset.i === String(p.r)));
}
function hide() {
  tip.style.opacity = 0; ring.style.opacity = 0; cur = null;
  if (links) while (links.firstChild) links.removeChild(links.firstChild);
  rows.forEach(r => r.classList.remove('on'));
}

box.addEventListener('pointermove', e => {
  const r = box.getBoundingClientRect();
  const p = nearest(e.clientX - r.left, e.clientY - r.top, r.width, r.height);
  if (p) show(p, e.clientX - r.left, e.clientY - r.top); else hide();
});
box.addEventListener('pointerleave', hide);

// 表の行にカーソルを乗せても，図の上の点が光る（逆引き）。
// **カーソルが使えない人にも同じ情報が届くように**，表を必ず添える
// （点が数千を超える図だけは表を絞る。絞ったことは図の下に明記する）。
rows.forEach(r => {
  r.addEventListener('mouseenter', () => {
    // 表の行は論理点。図の上では**先頭の面**の点を光らせる
    const p = PTS[Number(r.dataset.i)];
    if (!p) return;
    const w = box.clientWidth, h = box.clientHeight;
    show(p, p.x * w, p.y * h);
  });
  r.addEventListener('mouseleave', hide);
});

// 絞り込み。語・作品・時代のどれでも当たる
const q = document.getElementById('q');
if (q) q.addEventListener('input', () => {
  const s = q.value.trim();
  let n = 0;
  rows.forEach(r => {
    const hit = !s || r.textContent.includes(s);
    r.style.display = hit ? '' : 'none';
    if (hit) n++;
  });
  // 表を絞った図では，**表に無い語も図の上では当たる**。
  // 表の件数だけを出すと「無い」と誤解されるので両方を出す。
  const nlog = Number(document.body.dataset.nlog || rows.length);
  let extra = '';
  if (s && rows.length < nlog) {
    const seen = new Set();
    for (const p of PTS) if (p.term.includes(s)) seen.add(p.r);
    extra = '（図の上 ' + seen.size + ' 件）';
  }
  document.getElementById('count').textContent = n + ' 件' + extra;
  // **図の上にも印を付ける。** 表だけ絞っても「どこにあるか」は分からない。
  if (!marks) return;
  while (marks.firstChild) marks.removeChild(marks.firstChild);
  if (!s) return;
  const NS = 'http://www.w3.org/2000/svg';
  const w = box.clientWidth, h = box.clientHeight;
  let drawn = 0;
  for (const p of PTS) {
    if (!p.term.includes(s)) continue;
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', p.x * w); c.setAttribute('cy', p.y * h);
    c.setAttribute('r', 7);
    marks.appendChild(c);
    if (++drawn > 400) break;          // 印が多すぎると図が読めない
  }
});
"""


def save_interactive(fig, ax, stem, xs, ys, tips, out=None, title='',
                     note='', table_cols=None, source=None, id_col='語',
                     hint='', table_idx=None):
    """SVG を保存し，**同じ SVG を埋め込んだ対話的な HTML** も書く。

    ``xs`` ``ys`` はデータ座標，``tips`` は点ごとの情報
    （``{'term': 語, 'fields': [(見出し, 値), …]}`` の並び）。
    3つの長さは一致していなければならない。ずれたまま描くと，
    **指した点と出る語が食い違う**（注記の添字ずれと同じ事故）。

    位置は「図全体に対する割合」で書き出す。SVG を ``width:100%`` で
    伸縮させても割合は変わらないので，どんな幅でも点と当たり判定が
    合う。座標は matplotlib の変換を通して得るので，**図と HTML で
    座標の計算が二重にならない**。

    ``source`` に入力ファイルのパスを渡すこと。**どの表から描いた図かを
    HTML の末尾に刻む。** これが無いと，試験用の作りかけのデータから
    描いた図と，本番のデータから描いた図が見分けられない。
    入力がプロジェクトの外（``/tmp`` など）にあるときは
    「試験用」と赤字で出し，配布してはいけないことを図自身に言わせる。

    ``id_col`` は表の第1列の見出し（既定「語」。作品を点にする図では
    「作品」などに変える）。

    ``ax`` には**面の並び**も渡せる（``[axes[0], axes[1]]``）。同じ点を
    別の塗り分けで2面に描いた図では，どちらの面を指しても同じ情報が出る。
    表の行は点ごとに1行だけ作る（面の数だけ重複させない）。

    ``table_idx`` は**表に載せる点の添字**（既定は全点）。点が数千を超える
    図では表を全件出すと HTML が数 MB になり，読む側にも役に立たない。
    そのときは載せる点を選ぶ。**ただし図の当たり判定と検索は全点に効く**
    ので，表に無い語も指せるし検索で図に印が付く。表を絞ったときは，
    何件のうち何件を載せたかを HTML に明記する（黙って捨てないこと）。
    """
    import json as _json
    if not (len(xs) == len(ys) == len(tips)):
        raise ValueError(
            f'save_interactive: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'情報={len(tips)}）。座標と情報を同じ添字で絞り込むこと。')

    svg_path = save_fig(fig, stem, out=out)
    outdir = svg_path.parent

    # ---- 埋め込む SVG は**切り取らずに**保存する ------------------------
    # save_fig は bbox_inches='tight' で余白を詰めるため，図全体に対する
    # 割合と，ファイルの座標系がずれる。埋め込み用は詰めずに出す。
    import io
    buf = io.StringIO()
    fig.savefig(buf, format='svg', dpi=RASTER_DPI)
    svg = buf.getvalue()
    svg = svg[svg.index('<svg'):]          # XML 宣言と DOCTYPE を落とす

    # ---- 点の位置を図全体に対する割合で得る -----------------------------
    axes_list = list(ax) if isinstance(ax, (list, tuple, np.ndarray)) else [ax]
    W, H = fig.bbox.width, fig.bbox.height
    n_pts = len(tips)

    # **見出しは点ごとに書かない。** 点が1万個ある図では，
    # 「品詞」「頻度」…という見出しを1万回繰り返すだけで HTML が
    # 1 MB 以上ふくらむ。全点で見出しが同じなら1回だけ書き，
    # 値の並びだけを点に持たせる（JS 側で組み直す）。
    keys = [str(k) for k, _ in tips[0].get('fields', [])] if tips else []
    same_keys = bool(keys) and all(
        [str(k) for k, _ in t.get('fields', [])] == keys for t in tips)
    nheads = {str(t['notes'][0]) for t in tips if t.get('notes')}
    nhead = next(iter(nheads)) if len(nheads) == 1 else ''
    # 本文を渡さず ``notes=(見出し, None)`` としたときは，
    # ``links`` の先の語を JS 側で並べる
    linknotes = bool(nhead) and any(
        t.get('notes') and t['notes'][1] is None and t.get('links')
        for t in tips)

    pts = []
    for k, axk in enumerate(axes_list):
      pxy = axk.transData.transform(np.column_stack([np.asarray(xs, float),
                                                     np.asarray(ys, float)]))
      for j, (t, (px, py)) in enumerate(zip(tips, pxy)):
        i = k * n_pts + j
        # **変数名に注意。** ここを d と書くと，上で取った出力先 d
        # （svg_path.parent）を上書きして，最後に d / '....html' が
        # 「dict ÷ str」になる。実際に踏んだ。名前は使い回さない。
        # 添字 i は JS では使わない（行は r で引く）。点が1万個ある図では
        # 使わない値も 100 KB 単位で効くので書かない。
        rec = {'r': j, 'term': str(t.get('term', '')),
               'x': round(float(px) / W, 6),
               'y': round(1 - float(py) / H, 6)}        # SVG は上が 0
        if same_keys:
            rec['v'] = [str(b) for _, b in t.get('fields', [])]
        else:
            rec['fields'] = [[str(a), str(b)] for a, b in t.get('fields', [])]
        if t.get('notes'):
            # ('見出し', '本文') の2つ組。横に長い情報（近傍語など）。
            # 本文を None にすると，**線で結ぶ先の語を JS が並べる**
            # （同じ語の列を点ごとに書かずに済む。1万点で 1 MB 近く効く）
            if t['notes'][1] is None:
                pass
            elif nhead:
                rec['n'] = str(t['notes'][1])
            else:
                rec['notes'] = [str(t['notes'][0]), str(t['notes'][1])]
        if t.get('links'):
            # 原空間での近傍の添字。**同じ面の中で**線を結ぶ
            rec['links'] = [k * n_pts + int(q) for q in t['links']]
        pts.append(rec)

    cols = table_cols or keys
    head = f'<tr><th>{_esc(id_col)}</th>' + ''.join(
        f'<th>{_esc(c)}</th>' for c in cols) + '</tr>'
    # 数字の列だけ右寄せにする。時代名や作品 ID を右寄せにすると読みにくい。
    def _numish(v):
        t = str(v).strip().replace('%', '').replace(',', '')
        t = t.lstrip('+-')
        return bool(t) and t.replace('.', '', 1).isdigit()

    # 表に載せる点。**面の数だけ重複させない**（論理点1つに1行）
    if table_idx is None:
        order = list(range(n_pts))
    else:
        seen, order = set(), []
        for i in [int(q) for q in table_idx]:   # 重複を除きつつ順序は保つ
            if 0 <= i < n_pts and i not in seen:
                seen.add(i); order.append(i)

    # 表は **tips から作る**（点の JSON は見出しを省いてあるので）
    body = []
    for j in order:
        fv = {str(a): str(b) for a, b in tips[j].get('fields', [])}
        tds = ''
        for c in cols:
            v = fv.get(c, '')
            cls = ' class="num"' if _numish(v) else ''
            tds += f'<td{cls}>{_esc(v)}</td>'
        body.append(f'<tr data-i="{j}">'
                    f'<td>{_esc(tips[j].get("term", ""))}</td>{tds}</tr>')

    # ---- 由来を図自身に刻む -------------------------------------------
    # **どの表から描いた図かが分からないと，試験用のデータで描いた図が
    # 本物として配られる。** 実際に起きた（2026-09-22）。
    import datetime as _dt
    stamp = _dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M')
    src = Path(source) if source else None
    prov = f'点 {len(pts)} 個／作図 {stamp}'
    warn = ''
    if src is not None:
        try:
            mt = _dt.datetime.fromtimestamp(src.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        except OSError:
            mt = '不明'
        prov = f'入力 {src.name}（更新 {mt}）／' + prov
        # プロジェクトの外（/tmp など）から描いた図は試験用である
        try:
            outside = not str(src.resolve()).startswith(str(ROOT.resolve()))
        except Exception:                               # noqa: BLE001
            outside = True
        if outside or '/tmp/' in str(src):
            warn = ('<p class="danger">⚠ <b>試験用の入力から作った図である。'
                    f'配布してはいけない。</b>（入力 {_esc(str(src))}）</p>')
            prov = f'入力 {_esc(str(src))}／' + f'点 {len(pts)} 個／作図 {stamp}'

    html = f"""<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{_esc(title or stem)}</title>
<style>{INTERACTIVE_CSS}</style></head>
<body data-nlog="{n_pts}" data-ntable="{len(body)}"><div class="wrap">
<h1>{_esc(title or stem)}</h1>
<p class="sub">{_rich(note)}</p>
{warn}
<div class="card">
  <div class="figbox">
    {svg}
    <svg id="links"></svg><svg id="marks"></svg>
    <div id="hit"></div><div id="ring"></div><div id="tip"></div>
  </div>
  <p class="hint">{_rich(hint or '点にカーソルを近づけると語が出る（最も近い点を拾うので，真上に置かなくてよい）。図の中の注記は静止版と同じものである。')}</p>
  <div class="bar">
    <input id="q" type="search" placeholder="語・作品・時代で絞り込む">
    <span id="count">{len(body)} 件</span>
    <span>·</span>
    {(f'<span>表は {len(body)} 件（図の点は {n_pts} 件。'
      '表に無い語も図の上で指せる。検索は図の印にも効く）</span><span>·</span>')
     if len(body) < n_pts else ''}
    <a href="{_esc(svg_path.name)}" download>SVG を保存</a>
    <span>（この HTML の中の図はその SVG そのもの）</span>
  </div>
  <div class="scroll"><table><thead>{head}</thead>
    <tbody>{''.join(body)}</tbody></table></div>
  <p class="prov">{prov}</p>
</div>
<script type="application/json" id="pts-data">{_json.dumps(
    {'keys': keys if same_keys else [], 'nhead': nhead,
     'linknotes': linknotes, 'pts': pts},
    ensure_ascii=False, separators=(',', ':'))}</script>
<script>{INTERACTIVE_JS}</script>
</div></body></html>
"""
    path = outdir / f'{stem}.html'
    path.write_text(html, encoding='utf-8')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB・対話版／'
          f'点 {len(pts)} 個)')
    return svg_path, path


def _esc(s):
    """HTML の特殊文字を落とす。**語はコーパス由来なので必ず通す。**"""
    return (str(s).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


def _rich(s):
    """注記の ``**…**`` だけを太字にする。

    説明文をノートブックと同じ書き方（Markdown 風）で書けるようにする。
    **先に必ず _esc を通す**ので，タグを書き込まれる余地は無い。
    ``**`` のままだと HTML では記号がそのまま出て読みにくい。
    """
    import re as _re
    return _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', _esc(s))


PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
           '#56B4E9', '#D55E00', '#F0E442', '#666666']

# --- 順序のあるものを塗るための1色相のランプ -----------------------------
# **時代・年次・段階のように順序のあるものを，上の8色で塗ってはいけない。**
# 明治中期が青で明治後期が黄なら，隣り合う時代が隣り合う色にならず，
# 「時代が下るとどちらへ動くか」という肝心のことが読めなくなる。
# 1色相の濃淡にすれば，近いもの同士が近い色になり，勾配がそのまま見える。
# 散布図の点は白地の上に置くので，いちばん明るい段は 100 ではなく
# 250（背景との対比 2:1）から始める。100 は面で塗るとき（ヒートマップ）用。
SEQ_BLUE_STEPS = ['#86b6ef', '#5598e7', '#3987e5',
                  '#256abf', '#184f95', '#0d366b']
SEQ_BLUE = LinearSegmentedColormap.from_list('jlit_blue', SEQ_BLUE_STEPS)

# 離散の順序（4区分など）を塗るときはこちら。隣の段と明度差が十分あり，
# いちばん明るい段も背景から浮く（対比 2:1 以上）ことを確かめてある。
SEQ_BLUE_5 = ['#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b']

# 大分類の2色。散布図ではどの2点も隣り合いうるので**全ペアが
# 見分けられる必要**があり，使える色数は多くない。2色に絞って，
# 下位の区別は印の形に持たせる。
GENRE_C = {'Fiction': '#2a78d6', 'Nonfiction': '#eb6834'}

# --- 初出年の5段 ---------------------------------------------------------
# 切れ目は period と同じ 1900／1912／1926／1945。
# **6段にはできない。** 1色相の濃淡で順序を見せるには，隣り合う段の明度差が
# 0.06 以上要る。この青系ランプは 250→700 で明度差にして 0.30 ほどしか幅が
# 無いので，段を6つ取るとどこかが 0.05 台に落ち，隣が見分けられなくなる。
# そこで作品数3点の明治前期（〜1886）を明治中期にまとめて5段とする。
YEAR_EDGES = [1900, 1912, 1926, 1945]
YEAR_LABELS = ['〜1899 明治前・中期', '1900-1911 明治後期', '1912-1925 大正',
               '1926-1944 昭和戦前', '1945- 昭和戦後']


def year_bands(years):
    """初出年を5段に畳み，``(段番号, ラベル, 色)`` を返す。

    段番号は 0〜4。**初出年が読めないものは -1** にする。0 に落とすと
    年の分からない作品が全部いちばん古い段に入り，通時の議論が崩れる。

    時代で塗る図はすべてこれを通すこと。同じ色が全ステップで同じ時代を
    指すようになり，Step 1 の図と Step 7 の図を並べて読める。
    """
    y = pd.to_numeric(pd.Series(list(years)), errors='coerce')
    code = np.full(len(y), -1, dtype=int)
    ok = y.notna().values
    if ok.any():
        code[ok] = np.digitize(y[ok].values, YEAR_EDGES)
    return code, YEAR_LABELS, SEQ_BLUE_5


def run_script(script, *args, tail=4000):
    """scripts/ のスクリプトを実行し，標準出力・標準エラー・終了コードを必ず表示する。

    print(r.stdout or r.stderr) では，標準出力が空でないときに
    エラーの内容が隠れてしまう。学習用には両方見えるほうがよい。
    """
    cmd = [sys.executable, str(ROOT / 'scripts' / script)] + [str(a) for a in args]
    print('$ python', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-tail:])
    if r.stderr.strip():
        print('--- stderr ---')
        print(r.stderr[-tail:])
    print(f'[exit {r.returncode}]')
    return r


# 自分の作業フォルダ（共用 iMac では必ず自分の名前で作ること）
ME = os.environ.get('JLIT_USER', 'student')
# 使うメタデータ。増補分（45点）を含む v3 があればそちらを優先する。
# v2 は v1 の 64 点しか無いので，増補後のコーパスで v2 を使うと
# 突合が外れて period も genre も空になる（Step 3 で v3 を作る）。
META = ROOT / 'metadata' / 'corpus_metadata_v3.csv'
if not META.exists():
    META = ROOT / 'metadata' / 'corpus_metadata_v2.csv'

OUT = ROOT / 'results' / ME
OUT.mkdir(parents=True, exist_ok=True)
print('OUT  =', OUT)


## 1. 環境チェック

まず全員が次のセルを実行する。`ALL OK` が出るまで先に進まない。
足りないものがあれば，表示された指示のとおりに導入する
（詳細は `docs/00_setup_students.md`）。

macOS なら，たいていの不足は次の1行で解決する。**`sudo` は要らない。**

```bash
bash scripts/00_bootstrap_mac.sh             # DH Lab 共用 iMac
bash scripts/00_bootstrap_mac.sh --personal  # 自分の Mac
```

リポジトリは **`~/Documents/dh_project/JLit_Corpus_2026`** に clone し，
仮想環境は **`~/Documents/dh_project/.venv`** に置く（スクリプトが作る）。
下のセルの出力で「リポジトリ構成」と「仮想環境」がこの場所を指しているか，
カーネルが **Python (JLit)** かを確かめること。別の場所（古いコピーなど）を
指していたら，Jupyter をリポジトリで起動し直す。

### 共用 iMac を使う人へ — 表示される「作業中のマシン」を控えること

DH Lab の iMac は XCreds 認証で，**ホームはログインしたマシンにしか残らない**。
火曜に3号機で作った仮想環境と成果物は，木曜に5号機にログインしても無い。
故障ではなく，そういう仕組みである。

- 出力の先頭に出る**マシン名をレポートに控える**
- 成果物は毎回 `git push` して持ち運ぶ（`docs/00_setup_students.md` §5）
- 機体を移ったら上のスクリプトを再実行する。JDK・UniDic などは
  `/Users/Shared/jlit` に残っているので，その機体で誰かが済ませていれば
  仮想環境を作るだけで終わる

In [ ]:
r = run_script('00_env_check.py')

## 2. メタデータを読む

`metadata/corpus_metadata_v2.csv` は，v1 の `コーパスdescription.xlsx` を
青空文庫の図書カードに突き合わせて作り直したものである。

**v1 の何が問題だったか**（`changes_from_v1` シートに全件）:

| 列 | 問題 | 例 |
|---|---|---|
| `year` | 底本の刊年を初出年にしていた | 小栗虫太郎『潜航艇「鷹の城」』1977 → 実は1935 |
| `ndc` | NDC 番号ではなくラベル文字列 | 「小説、物語」→ 913 |
| `genre` | 英語・日本語・別題・誤綴の混在 | "Histrical novel", 「（二十世紀鉄仮面）」 |
| `comments` | 語り・形態・文体が無統制に混在 | 「現代、独白」「Collection」「Colloquial」 |
| `brow` | high/low の二値 | 児童書とノンフィクションが押し込まれていた |

**データを作る人は，列の意味を1行で説明できなければならない。**
説明できない列は，必ずあとで誤用される。

In [ ]:
# load_meta() は分析に使わない行（superseded / too_short）を落として読む。
# 生の表がほしいときは load_meta(analysis_only=False)。
meta = load_meta()
print('分析対象:', meta.shape)
meta[['id','author_ja','title_aozora','year_first','period','ndc',
      'genre_sub','narration','style_class','tokens']].head(12)

## 3. 演習 1 — 偏りを数える

次のセルを実行し，**どの軸がいちばん偏っているか**を自分の言葉で述べよ。
`value_counts()` は比率も出せる（`normalize=True`）。

In [ ]:
COLS = ['period','style_class','kana_orthography','ndc',
        'genre_main','audience','register_level','narration','author_sex']

# まず**偏りの一覧**を1枚で見る。区分がいくつあり，最大の区分が
# 何割を占めるか。ここが 0.5 を超える項目は，その項目で群を比べると
# 片方がほとんど無い状態で比べることになる。
summ = []
for col in COLS:
    vc = meta[col].value_counts()
    summ.append({'項目': col, '区分数': len(vc),
                 '最大の区分': str(vc.index[0]) if len(vc) else '',
                 '最大の件数': int(vc.iloc[0]) if len(vc) else 0,
                 '最大の割合': float(vc.iloc[0]/vc.sum()) if len(vc) else np.nan,
                 '欠損': int(meta[col].isna().sum())})
show(pd.DataFrame(summ).sort_values('最大の割合', ascending=False),
     caption=f'メタデータの偏り（分析対象 {len(meta)} 点）',
     fmt={'最大の割合': '{:.1%}'})

# 続いて内訳。項目名は各ブロックの先頭行だけに出す（表として読みやすい）
rows = []
for col in COLS:
    vc = meta[col].value_counts()
    rt = meta[col].value_counts(normalize=True)
    for i, k in enumerate(vc.index):
        rows.append({'項目': col if i == 0 else '', '区分': str(k),
                     '作品数': int(vc[k]), '割合': float(rt[k])})
show(pd.DataFrame(rows), caption='各項目の内訳', fmt={'割合': '{:.1%}'})

In [ ]:
# 語数ベースでも見る。作品数と語数で印象が変わる軸はどれか。
g = (meta.groupby('period')
        .agg(works=('id','count'), tokens=('tokens','sum'))
        .assign(work_pct=lambda d: d.works/d.works.sum(),
                token_pct=lambda d: d.tokens/d.tokens.sum()))
show(g.reset_index().rename(columns={'period':'時代','works':'作品数',
                                      'tokens':'語数','work_pct':'作品数の割合',
                                      'token_pct':'語数の割合'}),
     caption='時代の構成 — 作品数で見るか語数で見るか',
     fmt={'語数':'{:,.0f}','作品数の割合':'{:.1%}','語数の割合':'{:.1%}'})

fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(len(g))
ax.bar(x-0.2, g.work_pct, .38, label='作品数の比率', color=PALETTE[0])
ax.bar(x+0.2, g.token_pct, .38, label='語数の比率', color=PALETTE[1])
ax.set_xticks(x); ax.set_xticklabels([i.split('_',1)[1] for i in g.index],
                                      rotation=20, ha='right')
ax.set_ylabel('比率'); ax.legend(frameon=False)
ax.set_title('時代区分の代表性：作品数 vs 語数')
ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=.25)
fig.tight_layout(); save_fig(fig, 'Step1_period_balance'); plt.show()

### 図は SVG で保存する

`save_fig()` は図を **SVG（ベクタ形式）**で `results/<自分の名前>/` に書き出す。
PNG ではない。理由は3つ。

1. **拡大しても劣化しない。** PNG は画素の並びなので，スライドで投影したり
   論文に載せたりすると文字が潰れる。SVG は輪郭の記述なので何倍にしても鮮明
2. **あとから直せる。** Illustrator や Inkscape で開いて，軸ラベルの位置や
   凡例の文字だけを直せる。図を作り直す必要がない
3. **査読・投稿に通る。** 多くの学術誌がベクタ形式を要求する

既定では `svg.fonttype='path'`，つまり**文字をアウトライン（図形）に変換**する。
日本語フォントの入っていない環境で開いても崩れないためである。
編集しやすさを優先するなら `plt.rcParams['svg.fonttype'] = 'none'` にすると
文字が `<text>` 要素のまま残るが，閲覧側に同じフォントが必要になる。

点が数千個ある散布図では，**散布図だけ** `rasterized=True` を指定する。
点はラスタ化されてファイルが軽くなり，軸と文字はベクタのまま残る。

> **画面に出る図と，保存される図は別物である。**
> ノートブックの中に表示される図は PNG（Jupyter や VS Code の版によっては
> SVG がそのまま表示されないことがあるため）。**保存されるファイルは SVG**
> なので，レポートに貼るほうはベクタである。拡大して確かめたいときは，
> `save_fig` が表示するパスの `.svg` をブラウザで開くこと。

### セルを実行しても何も出ないとき

ノートブックのセルは，必要な入力が無ければ `need()` が理由を表示して
何もしない。たとえば：

```
[未実行] data/datasets/chunks_index.csv がありません。
         先に 06_build_datasets.py のセルを実行すること
```

これは**壊れているのではなく，前の工程がまだ走っていない**という意味である。
表示された指示に従い，上のセルから順に実行し直すこと。

## 4. 演習 2 — 文語と口語の連続体

`bungo_per10k`（なり・けり・べし・ごとし…）と `kogo_per10k`（です・ます・である…）を
散布図にする。**言文一致運動の前後**を1枚で見せる図になるはずだが……

### 時代は「カテゴリ」ではなく「順序」である

`period` を凡例に取って8色で塗り分けるのが素朴なやり方だが，それでは
**明治中期が青，明治後期が黄，大正が赤**…となって，隣り合う時代が
隣り合う色にならない。この図で見たいのは個々の時代の位置ではなく
**時代が下るにつれて点がどちらへ動くか**だから，それでは肝心のものが
見えない。順序のあるものは**1色相の濃淡**に割り当てる。

段数は**5段**とする。6段にはできない。1色相の濃淡で順序を見せるには
隣り合う段の明度差が 0.06 以上要るが，白地の散布図で使える青の幅
（背景から浮く 250 から最も濃い 700 まで）は明度差にして 0.30 ほどしか
ない。6段取るとどこかが 0.05 台に落ち，隣の段と見分けられなくなる。
そこで作品数3点の**明治前期（〜1886）を明治中期にまとめて**5段にする。

色だけでは「どちらへ動いたか」は意外と読み取れないので，
**各段の中央値を結んだ軌跡**を重ねる。平均ではなく中央値にするのは，
文語標識が桁で外れる作品（『たけくらべ』など）に平均が引きずられるため。

> ### ⚠ この軸の値は「辞書に依存する実測値」である
>
> `bungo_per10k` / `kogo_per10k` は，**トークン列から数えた実測値**である
> （メタデータに人が書き入れた値ではない）。文語助動詞を1語と切るか
> 2語に割るかは辞書によって違うので，**辞書を替えるとこの図は動く**。
> `style_class`（A_文語体 / B_過渡 / C_口語体）は `bungo_per10k` の閾値で
> 決めているから，**作品の所属が変わることもある**。
>
> 本コーパスの辞書は `unidic-novel`（2026-09-22 決定）。辞書を替えたら
> `00_extend_metadata.py --remeasure-all` で測り直し，**この図も描き直す**
> こと。→ Step 3 §3，`docs/dictionary_comparison.md` §8

In [ ]:
TOP_N = 10
top = meta.nlargest(TOP_N, 'bungo_per10k').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.6,6.2))

# ------------------------------------------------------------------
# **時代は順序のあるものである。** period をカテゴリの8色で塗ると，
# 明治中期が青で明治後期が黄…となり，隣り合う時代が隣り合う色にならない。
# そうすると「時代が下るにつれて点がどちらへ動くか」という，この図で
# 見たい当のものが読めなくなる。初出年を5段に畳み，1色相の濃淡に割り当てる
# （淡いほど古く，濃いほど新しい）。年の値から作るので，period が空でも
# 年さえあれば塗れる。
# ------------------------------------------------------------------
code, blabels, bcols = year_bands(meta.year_first)
if (code < 0).any():
    m = (code < 0)
    ax.scatter(meta.bungo_per10k[m], meta.kogo_per10k[m], s=46, alpha=.85,
               color='#c3c2b7', edgecolor='white', linewidth=.8,
               label=f'初出年不明（{int(m.sum())}点）', zorder=3)
for i, lab in enumerate(blabels):
    m = (code == i)
    if not m.any():
        continue
    # 濃淡5段を見分けさせるので，点は少し大きめにする。小さい点では
    # 面積が足りず，隣り合う段の明度差が目に入らない。
    ax.scatter(meta.bungo_per10k[m], meta.kogo_per10k[m], s=54, alpha=.9,
               color=bcols[i], edgecolor='white', linewidth=.8,
               label=f'{lab}（{int(m.sum())}点）', zorder=3)

# 各段の中央値をつなぐ。点の雲だけでは「どちらへ動いたか」は意外と
# 読み取れない。中央値の軌跡を1本引くと，時代による移動が線として出る。
# 中央値にするのは，文語標識が桁で外れる作品に平均が引きずられるため。
mx = [meta.bungo_per10k[code == i].median() for i in range(len(blabels))]
my = [meta.kogo_per10k[code == i].median() for i in range(len(blabels))]
ax.plot(mx, my, color='#55554f', linewidth=1.4, alpha=.85, zorder=4,
        label='各段の中央値（古→新）')
ax.scatter(mx, my, s=135, marker='D', c=bcols, edgecolor='#55554f',
           linewidth=1.2, zorder=5)
# 進む向きを矢印で示す。凡例を読まないと古新が分からない図は不親切である。
ax.annotate('', xy=(mx[-1], my[-1]), xytext=(mx[-2], my[-2]), zorder=6,
            arrowprops=dict(arrowstyle='-|>', color='#55554f', lw=1.4,
                            shrinkA=9, shrinkB=9))

ax.set_xlabel('文語助動詞標識（/万語・対数目盛）')
ax.set_ylabel('口語助動詞標識（/万語）')
ax.set_title('文語 ⇄ 口語（1点＝1作品／色＝初出年・濃いほど新しい）')
# linthresh を指定しないと目盛が 1 未満まで刻まれて右端が潰れる
ax.set_xscale('symlog', linthresh=10)
ax.grid(alpha=.25, linewidth=.6, zorder=0)
# 凡例は**順序どおり**に並べる。matplotlib は描いた順に並べるので，
# 淡→濃の順で描いておけば凡例もそのまま時代順になる。
ax.legend(frameon=False, fontsize=8.5, loc='upper left',
          bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
ax.spines[['top','right']].set_visible(False)

# **注記は tight_layout の後に置く。** 先に置くと軸が動いて位置がずれる。
fig.tight_layout()
reserve_right(fig, 0.82)      # 凡例を面の外に置いてあるため
# 作者名＋作品名をそのまま置くと，上位はどれも右下隅に固まっているので
# 6件中5件が重なって読めない。番号だけを打ち，名前は下の表で引く。
# 番号は 8pt。密集帯の点の間隔は最小 9px で，9pt の数字（幅 8.4px を
# 1.08 倍に見込む）では直上に載らない。8pt なら載る。
label_points(ax, top.bungo_per10k, top.kogo_per10k,
             [str(i+1) for i in range(len(top))], fontsize=8)

# 番号が付くのは上位10点だけである。残りの91点は「どの作品か」が
# 分からないまま眺めることになる。**対話版では全点を指して引ける。**
# 表と同じ情報を持たせるので，番号と表を往復する必要もなくなる。
# 図の番号は，語幹の突合ではなく**同じ基準で順位を振り直して**求める。
# top は reset_index してあるので語幹で引こうとすると空振りする
# （空振りしても例外は出ないので，全件の番号が黙って空になる）。
order = meta.bungo_per10k.rank(ascending=False, method='first')
tips = []
for pos, (_, r) in enumerate(meta.iterrows()):
    n = int(order.iloc[pos])
    tips.append({'term': f'{r.author_ja}『{r.title_aozora}』',
                 'fields': [('初出', f'{r.year_first:.0f}'
                                     if pd.notna(r.year_first) else '不明'),
                            ('時代', str(r.period)),
                            ('文体区分', str(r.style_class)),
                            ('文語標識', f'{r.bungo_per10k:.1f}'),
                            ('口語標識', f'{r.kogo_per10k:.1f}'),
                            ('正書法', str(r.kana_orthography)),
                            ('語数', f'{r.tokens:,.0f}'
                                     if pd.notna(r.tokens) else '—'),
                            ('図の番号', str(n) if n <= len(top) else '')]})
save_interactive(fig, ax, 'Step1_bungo_kogo',
                 meta.bungo_per10k, meta.kogo_per10k, tips,
                 source=META, id_col='作品',
                 title='文語 ⇄ 口語（1点＝1作品）',
                 note=('色＝初出年の5段（濃いほど新しい）／菱形と矢印は'
                       '各段の中央値／番号は文語標識の上位10点。'
                       '横軸は symlog（10 未満は線形）。'),
                 table_cols=['初出', '時代', '文体区分', '文語標識',
                             '口語標識', '正書法', '語数', '図の番号'])
plt.show()

tbl = top[['author_ja','title_aozora','year_first',
           'bungo_per10k','kogo_per10k','style_class']].copy()
tbl.insert(0, '順位', range(1, len(tbl)+1))
show(tbl.rename(columns={'author_ja':'作家','title_aozora':'作品',
                         'year_first':'初出','bungo_per10k':'文語標識',
                         'kogo_per10k':'口語標識','style_class':'文体区分'}),
     caption=f'文語標識の上位{TOP_N}件（図中の番号に対応・万語あたり）',
     fmt={'文語標識':'{:.1f}','口語標識':'{:.1f}','初出':'{:.0f}'})

### 考えてみよう

- 図の左下（文語も口語も少ない）に何があるか。それは何を意味するか。
- **中央値の軌跡はどちらへ向かっているか。**単調か，途中で折り返すか。
  折り返すとしたら，それは文体の変化か，それとも各段に入っている作品の
  顔ぶれ（ジャンル・作家）の違いか。
- `style_class` が `A_文語体` の作品は何点か。それで「言文一致以前」を代表できるか。
- **このコーパスで「言文一致による文体変化」を論じられるか。論じられないとしたら何が足りないか。**

`metadata/expansion_candidates.csv` に，この空白を埋めるための候補を挙げてある。

In [ ]:
cand = pd.read_csv(ROOT/'metadata'/'expansion_candidates.csv')

# priority は 1〜5 の整数のはずだが，手で編集される表なので
# 数値でない値が紛れることがある。そのまま cand.priority<=2 と書くと
# 列全体が文字列として読まれ TypeError で落ちる。数値化してから比べる。
cand['priority'] = pd.to_numeric(cand['priority'], errors='coerce')
bad = cand['priority'].isna().sum()
if bad:
    print(f'[warn] priority が数値でない行が {bad} 件ある（絞り込みから外れる）')

cols = ['priority','gap','author_ja','title','year_target','style_expect','rationale']
show(cand[cand['priority'] <= 2][cols].sort_values('priority').head(20),
     caption='増補の候補（優先度1・2のみ／上位20件）')

## 5. 演習 3 — 欠陥を自分で見つける

メタデータではなく**テクスト本体**を見る。検証スクリプトを走らせる前に，
まず素朴な方法で重複を探してみよう。

### 手がかり：統計量が近すぎるファイルはないか

In [ ]:
# v1 コーパス（64点の .txt）の場所。指定の仕方は2通りある。
#
#  (a) **このセルに直接書く** … 手っ取り早い。下の V1_PATH を埋める
#  (b) **環境変数で渡す**     … 機体を移っても効く。Jupyter を起動する
#      **前に**シェルで次を実行しておく
#          export JLIT_CORPUS_V1=~/Dropbox/Corpus/DH_text_analytics_2025/corpus
#
# **`os.environ.get()` の中に `export …` と書いてはいけない。** そこに入るのは
# 環境変数の**名前**であって，シェルのコマンドではない。書いても例外は出ず，
# 「そんな名前の変数は無い」と判定されて既定値に落ちるだけなので気づきにくい。
# 直接書きたいときは V1_PATH のほうを使うこと。
V1_PATH = ''        # 例: '~/Dropbox/Corpus/DH_text_analytics_2025/corpus'

# expanduser を通すこと。'~/…' は Path が展開しないので，そのままでは
# 「存在しない」と判定される。
CORPUS_V1 = Path(os.path.expanduser(
    V1_PATH or os.environ.get('JLIT_CORPUS_V1') or str(ROOT/'data'/'corpus_v1')))
print(f'v1 コーパス: {CORPUS_V1}')
print('  →', '見つかった' if CORPUS_V1.exists() else '**見つからない**')
if not CORPUS_V1.exists():
    print('  本文そのものを読む検査（次のセルと 99_validate）は飛ばされる。')
    print('  場所が分かっているなら，このセルの V1_PATH に書いて再実行すること。')

# ---- ここから下は診断表だけで動く。v1 コーパスが無くても実行できる -------
# 本文を読まずに重複を疑う，というのがこの演習の要点である。
d = pd.read_csv(ROOT/'metadata'/'diagnostics_v1.csv')
# 語数・異なり語数・漢字率が極端に近いペアを探す
cols = ['tokens','types','kanji_ratio']
X = d[cols].values.astype(float)
Xn = (X - X.mean(0)) / X.std(0)
D = np.linalg.norm(Xn[:,None,:]-Xn[None,:,:], axis=2)
np.fill_diagonal(D, np.inf)
i,j = np.unravel_index(np.argmin(D), D.shape)
show(d.iloc[[i,j]][['file']+cols],
     caption=f'最も統計量の近いペア（標準化距離 {D[i,j]:.5f}）',
     fmt={'tokens':'{:,.0f}','types':'{:,.0f}','kanji_ratio':'{:.3f}'})
print('**距離がほぼ 0 なら同一本文の疑い。** 次のセルで n-gram で確かめる。')

In [ ]:
# 8-gram シングルによる重複検出（99_validate.py の中核）
def shingles(text, n=8, step=3):
    t = text.split()
    return {tuple(t[i:i+n]) for i in range(0, max(0,len(t)-n), step)}

if CORPUS_V1.exists():
    a = (CORPUS_V1/d.file[i]).read_text(encoding='utf-8')
    b = (CORPUS_V1/d.file[j]).read_text(encoding='utf-8')
    A,B = shingles(a), shingles(b)
    print(f'8-gram 包含率 = {len(A&B)/min(len(A),len(B)):.1%}')
    print('\n--- 冒頭120字 ---')
    print('A:', a[:120].replace(chr(10),'/'))
    print('B:', b[:120].replace(chr(10),'/'))

### 何が起きていたか

`乱歩_灰色の巨人.txt` の本文は **『魔法博士』と同一**である。章題まで一致する
（動く映画館／悪魔の国／人造人間／黄金怪人／井戸の中から／奇々怪々…）。
つまり実効サンプル数は 64 ではなく **63**。

さらに 2 種類の欠陥がある。

1. **外字の喪失** — 青空文庫の外字注記 `※［＃「…」、第4水準2-81-40］` から
   `［＃…］` だけを削った結果，`※` が本文に残り，文字が失われている。
   全体で **592箇所**。『不如帰』の「合※の式」はもと「合巹の式」である。
2. **奥付の混入** — `海野十三_敗戦日記.txt` の末尾に
   「入力：青空文庫／校正：伊藤時也／ファイル作成：野口英司」以下がそのまま残っている。

いずれも「正規表現で要らないものを削る」という方針の副作用である。
Step 2 からは **削らずにタグで分離する** 方針に切り替える。

In [ ]:
# 全件検査。FATAL が出るのが正しい（これが Step 2–3 で直す対象）
#
# **ここで渡すメタデータは v2 である。** v3 は Step 3 で作り直した
# 108 点のコーパスを記述する表で，v1 の欠陥（本文の取り違え・不完全収録）
# はすでに解消済みとして書いてある。v3 を渡すと「FATAL が出るのが正しい」
# はずの検査が何も出さず，何を直したのかが分からなくなる。
META_V1 = ROOT/'metadata'/'corpus_metadata_v2.csv'
if CORPUS_V1.exists():
    run_script('99_validate.py', '--corpus', CORPUS_V1,
               '--meta', META_V1,
               '--out', OUT/'Step1_validation.csv')
else:
    need(CORPUS_V1, 'v1 コーパスの場所を環境変数 JLIT_CORPUS_V1 で指定すること')

## 6. このステップの課題

`results/<自分の名前>/Step1_report.md` に次を書いて提出する（600–1000字）。

1. このコーパスで**最も深刻な偏り**はどれか。作品数と語数の両方を根拠に述べよ。
2. その偏りは，どんな研究上の問いを**不可能にする**か。具体的に1つ挙げよ。
3. `expansion_candidates.csv` の **`in_corpus` が `未収録` の行**から3点選び，
   なぜその3点かを説明せよ。11件のうち**4件は `保護期間中`**（著作権存続）である。
   「入れたい作品」と「入れられる作品」が一致しないことが何を意味するか，
   1 で述べた偏りと結びつけて論じること。
4. 図を最低1枚（自分で作ったもの）添付せよ。**SVG で提出すること。**

### このステップの到達点（次へ進む条件）

- `00_env_check.py` が `ALL OK` を返す（Python・UniDic・MALLET・日本語フォント）
- 表示された**マシン名を控えた**（共用 iMac の場合）
- `save_fig()` で SVG を書き出し，ブラウザで開いて日本語が読めることを確認した
- その図を含めて `git push` できた
- v1 の偏りを示す図を自分で1枚作った
- 3つの重大な欠陥を自分の手で再発見した
- `docs/representativeness_report.md` を通読した
